# SmolVLA on LIBERO-10 with `lerobot-eval` (Kaggle, 2 x T4)

Runs the protocol of `baseline_performance/results_300926/` through `baseline_performance/evaluate_all.py`:
10 tasks x seeds 0, 1, 2 x 10 episodes (init states 0-9), each (task, seed) in its own `lerobot-eval`
process, several at a time across both GPUs. Step 9 compares the result per task and per init state with
the local run and with the external LIBERO-CTRL reference.

**Which checkpoint** (`CHECKPOINT` in step 1):

- `"lerobot"` (default): `lerobot/smolvla_libero`, the checkpoint LIBERO-CTRL evaluated at 56.8% on LIBERO-10
  (`baseline_performance/reference_libero_ctrl/`). Tests whether our evaluation path reproduces that
  number: if it does, the gap between our 40.7% and the published 71% is the checkpoint, not the harness.
- `"HuggingFaceVLA"`: `HuggingFaceVLA/smolvla_libero`, the checkpoint behind `results_300926` (reproduced
  on Kaggle in `week6/baseline_lerobot_eval_kaggle/`).
- `"continued"`: `lerobot/smolvla_libero` after continued training on LIBERO-10
  (`continued_training/README.md`). Upload the `smolvla_libero10_continued_final/` folder from Google Drive as a
  private Kaggle dataset and add it as input (**Add Input > Datasets > Your Datasets**). Step 9 then also
  compares it episode by episode with the `lerobot/smolvla_libero` run of this notebook
  (`week6/smolvla_libero_lerobot_eval_kaggle/`, 171/300), which is the test of the hypothesis.

**Step 0 of `noise_reachability/`** (other `n_action_steps`): set `CHECKPOINT = "lerobot"`,
`N_ACTION_STEPS = 10` (then 50, as a second version) and `TASKS = "0 4 6 8"` in step 1. Results go to
`smolvla_libero_lerobot_eval_nas<N>/`, and step 9 compares them with the `n_action_steps=1` run
(`week6/smolvla_libero_lerobot_eval_kaggle/`) on the same episodes.

**Step 1 of `noise_reachability/`** (more noise seeds), two versions. Set only `RUN_TAG` in step 1; it fixes
`CHECKPOINT = "lerobot"`, `N_ACTION_STEPS = 10`, `TASKS = "0 4 6 7 8"` and the seeds, and step 1 prints them:
- `RUN_TAG = "step1"`, seeds 3-15 (650 episodes). Step 9 then reports pass@k
  per init state together with seeds 0-2 from `week6/noise_reachability_step0_kaggle/nas10_all_tasks/`.
- `RUN_TAG = "step1_repeat"`, seed 0 (50 episodes): seed 0 run again, to measure run-to-run variation.

`RUN_TAG` gives each version its own output folder (`smolvla_libero_lerobot_eval_nas10_<RUN_TAG>/`).

Evaluations run seed by seed (all 10 tasks of seed 0 first), so a session that ends early still covers
every task once.

**Kaggle settings (right-hand panel), before running**

1. Accelerator: **GPU T4 x2**. Not P100: the pinned PyTorch build does not support it.
2. Internet: **On** (needs a phone-verified Kaggle account). Packages, the repo and the model are downloaded.

**How to run.** Use **Save Version > Save & Run All (Commit)**. The notebook then runs in the background with
the browser closed, for at most 12 hours, and everything in `/kaggle/working/baseline_lerobot_eval/` is kept as
the version's output. The run stops starting new evaluations after 9 hours and stops completely after 11, so
the output is always saved.

**If it does not finish in one session** (step 9 then says INCOMPLETE): open the notebook, **Add Input >
Your Work >** this notebook, select the last version's output, and Save & Run All again. Step 6 copies the
finished evaluations in and only the missing ones run.

Keep every evaluation of this run on Kaggle; do not pool its episodes with local or Colab ones. The two
checkpoints write to different output folders, so their resumes never mix.

## 1. Settings

In [ ]:
import csv, glob, json, os, re, shutil, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/swagatk/efficient_vla.git"
REPO_REF = "main"
REPO = "/tmp/efficient_vla"        # code and venv live outside /kaggle/working, so they are not saved as output
VENV = "/tmp/venv"
PY = f"{VENV}/bin/python"

CHECKPOINT = "lerobot"             # "lerobot", "HuggingFaceVLA" or "continued"; see the introduction
CHECKPOINTS = {
    # The checkpoint behind results_300926. Its config already has n_action_steps=1 and LIBERO camera names.
    "HuggingFaceVLA": dict(repo="HuggingFaceVLA/smolvla_libero", revision="6721902bc4d61e50a3bfdb11dfb4cb626f05d102",
                           n_action_steps=None, rename_map=None, out_name="baseline_lerobot_eval"),
    # The checkpoint LIBERO-CTRL evaluated. Its config defaults to n_action_steps=50; it was evaluated, and
    # trained, with one action per inference (n_action_steps=1). It names its cameras camera1/camera2.
    "lerobot": dict(repo="lerobot/smolvla_libero", revision="a24f1f380461b665158d52acd140152c79e8b36d",
                    n_action_steps=1,
                    rename_map={"observation.images.image": "observation.images.camera1",
                                "observation.images.image2": "observation.images.camera2"},
                    out_name="smolvla_libero_lerobot_eval"),
    # lerobot/smolvla_libero after continued training (continued_training/), read from a Kaggle dataset input.
    # Same evaluation settings as "lerobot"; its config keeps the starting checkpoint's n_action_steps=50.
    "continued": dict(repo=None, revision=None, job_name="smolvla_libero10_continued",
                      n_action_steps=1,
                      rename_map={"observation.images.image": "observation.images.camera1",
                                  "observation.images.image2": "observation.images.camera2"},
                      out_name="continued_libero10_eval"),
}
# noise_reachability step 1: setting RUN_TAG alone fixes the checkpoint, n_action_steps, tasks and seeds
# (a version run with only RUN_TAG and SEEDS changed got n_action_steps=1 and all 10 tasks).
RUN_TAG = ""                       # "step1" or "step1_repeat" for noise_reachability step 1; "" otherwise
STEP1_SEEDS = {"step1": "3 4 5 6 7 8 9 10 11 12 13 14 15", "step1_repeat": "0"}
if RUN_TAG in STEP1_SEEDS:
    CHECKPOINT = "lerobot"
CKPT = CHECKPOINTS[CHECKPOINT]
N_ACTION_STEPS = None              # None keeps the checkpoint entry's value; set for noise_reachability step 0
if RUN_TAG in STEP1_SEEDS:
    N_ACTION_STEPS = 10
if N_ACTION_STEPS is not None:
    CKPT = dict(CKPT, n_action_steps=N_ACTION_STEPS, out_name=f"{CKPT['out_name']}_nas{N_ACTION_STEPS}")
if RUN_TAG:
    CKPT = dict(CKPT, out_name=f"{CKPT['out_name']}_{RUN_TAG}")

OUT_NAME = CKPT["out_name"]
OUT_ROOT = f"/kaggle/working/{OUT_NAME}"  # results, kept as the version's output

TASKS = "0 1 2 3 4 5 6 7 8 9"     # "0 4 6 8" for noise_reachability step 0
SEEDS = "0 1 2"
if RUN_TAG in STEP1_SEEDS:
    TASKS, SEEDS = "0 4 6 7 8", STEP1_SEEDS[RUN_TAG]
print(f"checkpoint {CHECKPOINT}, n_action_steps {CKPT['n_action_steps']}, tasks {TASKS}, seeds {SEEDS}, output {OUT_NAME}")
N_EPISODES = 10                    # init states 0-9, the protocol of results_300926
ORDER = "seed"                     # all tasks of seed 0 first, then seed 1, then seed 2
LAUNCH_DEADLINE_MIN = 9 * 60       # Kaggle stops a session after 12 hours
HARD_STOP_MIN = 11 * 60
GB_PER_JOB = 5                     # one lerobot-eval process held about 4.5 GB of RAM locally

# Commits of the editable installs in the local lerobot_env that produced the existing results.
LEROBOT_COMMIT = "b6ec0060779550c0a157ae34feb89e0cf86012a8"
LIBERO_COMMIT = "8f1084e3132a39270c3a13ebe37270a43ece2a01"


def sh(cmd, check=True, env=None):
    # Capture and print: notebooks do not show output that a subprocess writes directly.
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", env=env, capture_output=True, text=True)
    print((r.stdout + r.stderr).rstrip(), flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")
    return r

## 2. Hardware

In [ ]:
smi = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv,noheader")
gpus = [line.split(", ") for line in smi.stdout.strip().splitlines()]
assert gpus, "No GPU. Set Accelerator to GPU T4 x2."
for name, mem, cap, driver in gpus:
    assert float(cap) >= 7.5, f"{name} (compute capability {cap}) is not supported by the pinned PyTorch build; use GPU T4 x2"
CUDA_DRIVER = float(re.search(r"CUDA Version: ([0-9.]+)", sh("nvidia-smi").stdout).group(1))
mem_gb = int(open("/proc/meminfo").read().split()[1]) / 1e6
avail_gb = int(re.search(r"MemAvailable:\s+(\d+)", open("/proc/meminfo").read()).group(1)) / 1e6
JOBS = max(1, min(2 * len(gpus), int((avail_gb - 4) // GB_PER_JOB), os.cpu_count()))
GPU_IDS = ",".join(str(i) for i in range(len(gpus)))
print(f"{len(gpus)} GPU(s), driver supports CUDA {CUDA_DRIVER}; {os.cpu_count()} CPU cores; "
      f"RAM {mem_gb:.0f} GB ({avail_gb:.0f} GB available) -> {JOBS} evaluations at a time on GPUs {GPU_IDS}")

## 3. Code

In [ ]:
if not Path(REPO, ".git").exists():
    sh(f"git clone {REPO_URL} {REPO}")
sh(f"git -C {REPO} fetch origin && git -C {REPO} checkout {REPO_REF} && git -C {REPO} pull --ff-only origin {REPO_REF}")
sh(f"git -C {REPO} log -1 --format='%H %s'")

## 4. Environment

A separate virtual environment with the package versions of the local `lerobot_env`. If the GPU driver is
too old for the CUDA 13 build of PyTorch, the same PyTorch version is installed in a CUDA 12 build instead
and the choice is printed. Takes several minutes.

In [ ]:
# Headless rendering: MuJoCo needs the NVIDIA EGL vendor file, which some images lack.
icd = Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not icd.exists():
    icd.parent.mkdir(parents=True, exist_ok=True)
    icd.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}))
sh("apt-get install -y -qq libegl1 libgles2 libgl1 > /dev/null", check=False)

sh("python -m pip install -q uv")
if not Path(PY).exists():
    sh(f"uv venv {VENV} --python 3.12")

req = Path(REPO, "hybrid_diffusion", "requirements-colab.txt").read_text().splitlines()
if CUDA_DRIVER >= 13.0:
    TORCH_BUILD = "cu130 (as pinned)"
    reqs = req
else:
    cu = "cu128" if CUDA_DRIVER >= 12.8 else "cu126"
    assert CUDA_DRIVER >= 12.6, f"driver supports only CUDA {CUDA_DRIVER}; need 12.6 or newer"
    TORCH_BUILD = f"{cu} (driver too old for cu130)"
    torch_pins = [l for l in req if l.split("==")[0] in ("torch", "torchvision")]
    sh(f"uv pip install -q --python {PY} " + " ".join(f"'{p}+{cu}'" for p in torch_pins)
       + f" --index-url https://download.pytorch.org/whl/{cu} --extra-index-url https://pypi.org/simple")
    reqs = [l for l in req if not re.match(r"(torch|torchvision|triton|nvidia-)", l)]
print("PyTorch build:", TORCH_BUILD)
Path("/tmp/requirements.txt").write_text("\n".join(reqs) + "\n")
sh(f"uv pip install -q --python {PY} --no-deps -r /tmp/requirements.txt")
sh(f"uv pip install -q --python {PY} --no-deps 'lerobot @ git+https://github.com/huggingface/lerobot@{LEROBOT_COMMIT}'")

LIBERO = os.path.expanduser("~/LIBERO")
if not Path(LIBERO, ".git").exists():
    sh(f"git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git {LIBERO}")
sh(f"git -C {LIBERO} checkout -q {LIBERO_COMMIT}")
Path(LIBERO, "libero", "__init__.py").touch()   # present in the local checkout, missing upstream
sh(f"uv pip install -q --python {PY} --no-deps -e {LIBERO}")

# LIBERO asks for these paths interactively on first import unless the config already exists.
root = f"{LIBERO}/libero/libero"
os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)
os.makedirs(f"{LIBERO}/libero/datasets", exist_ok=True)
Path(os.path.expanduser("~/.libero/config.yaml")).write_text(
    f"benchmark_root: {root}\nbddl_files: {root}/bddl_files\ninit_states: {root}/init_files\n"
    f"datasets: {LIBERO}/libero/datasets\nassets: {root}/assets\n"
)

## 5. Sanity checks

Checks that PyTorch sees both GPUs and can run bfloat16 on them (SmolVLA runs in bfloat16; the T4 has no
native support, so this can be slow but must work), and that MuJoCo can render headless. If EGL rendering
fails, the cell switches to CPU rendering (OSMesa), which is slower; the backend is recorded in
`provenance.jsonl`.

In [ ]:
RUN_ENV = dict(os.environ, PATH=f"{VENV}/bin:" + os.environ["PATH"], MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl",
               MPLBACKEND="Agg", WANDB_MODE="disabled", OMP_NUM_THREADS="1")

sh(f"""{PY} -c "
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
x = torch.randn(256, 256, device='cuda', dtype=torch.bfloat16)
print('bfloat16 matmul ok', float((x @ x).float().abs().mean()) > 0)
" """, env=RUN_ENV)

egl = sh(f"{PY} -c \"import mujoco; c = mujoco.GLContext(64, 64); c.make_current(); print('EGL context ok')\"", check=False, env=RUN_ENV)
if "EGL context ok" not in egl.stdout:
    print("EGL rendering failed; switching to OSMesa (CPU rendering, slower).")
    sh("apt-get install -y -qq libosmesa6-dev > /dev/null")
    RUN_ENV.update(MUJOCO_GL="osmesa", PYOPENGL_PLATFORM="osmesa")
    sh(f"{PY} -c \"import mujoco; c = mujoco.GLContext(64, 64); c.make_current(); print('OSMesa context ok')\"", env=RUN_ENV)
print("render backend:", RUN_ENV["MUJOCO_GL"])

if CKPT["repo"] is None:
    # A Kaggle dataset input: the one folder whose train_config.json names this training job.
    found = [str(Path(f).parent) for f in glob.glob("/kaggle/input/**/train_config.json", recursive=True)
             if json.load(open(f)).get("job_name") == CKPT["job_name"] and Path(f).with_name("model.safetensors").exists()]
    assert len(found) == 1, f"need exactly one input folder with job_name {CKPT['job_name']}, found {found}"
    POLICY_PATH = found[0]
    CKPT["repo"], CKPT["revision"] = POLICY_PATH, "kaggle-input"
    print("train_config:", {k: v for k, v in json.load(open(f"{POLICY_PATH}/train_config.json")).items()
                            if k in ("steps", "batch_size", "seed", "job_name")})
else:
    # Pinned checkpoint revision, downloaded once; lerobot-eval loads it from the local snapshot folder.
    r = sh(f"""{PY} -c "from huggingface_hub import snapshot_download; print(snapshot_download('{CKPT['repo']}', revision='{CKPT['revision']}'))" """, env=RUN_ENV)
    POLICY_PATH = r.stdout.strip().splitlines()[-1]
EVAL_ARGS = f"--policy_path {POLICY_PATH} --lerobot_eval {VENV}/bin/lerobot-eval --order {ORDER}"
if CKPT["n_action_steps"] is not None:
    EVAL_ARGS += f" --n_action_steps {CKPT['n_action_steps']}"
if CKPT["rename_map"]:
    EVAL_ARGS += f" --rename_map '{json.dumps(CKPT['rename_map'])}'"
print(f"checkpoint {CKPT['repo']}@{CKPT['revision'][:10]} -> {POLICY_PATH}")
print("evaluate_all.py arguments:", EVAL_ARGS)
sh(f"cd {REPO} && {PY} -m pytest tests/ -q -p no:cacheprovider", env=RUN_ENV)

## 6. Resume from an earlier version's output

If an earlier version's output was added as input, its finished evaluations are copied in. Nothing happens
on a first run.

In [ ]:
os.makedirs(OUT_ROOT, exist_ok=True)
for prev in sorted(glob.glob(f"/kaggle/input/**/{OUT_NAME}", recursive=True)):
    n = len(glob.glob(f"{prev}/task_*_seed_*/eval_info.json"))
    print(f"copying {n} finished evaluations from {prev}")
    shutil.copytree(prev, OUT_ROOT, dirs_exist_ok=True)
done = len(glob.glob(f"{OUT_ROOT}/task_*_seed_*/eval_info.json"))
print(f"{done} of {len(TASKS.split()) * len(SEEDS.split())} evaluations already finished")

## 7. Timing check (one episode, not counted)

One episode of task 3 in a separate folder. The base policy solves most task 3 episodes, so a failure here
is worth a look in `_timing/task_3_seed_0/eval.log`, but does not stop the run. Its `eval.log` also shows
that the checkpoint loaded with the intended camera names and `n_action_steps`. Skipped when resuming.

In [ ]:
if done == 0:
    t0 = time.time()
    sh(f"cd {REPO} && {PY} baseline_performance/evaluate_all.py --resume_dir {OUT_ROOT}/_timing --tasks 3 --seeds 0 "
       f"--n_episodes 1 {EVAL_ARGS} --hard_stop_min 60", check=False, env=RUN_ENV)
    print(open(f"{OUT_ROOT}/_timing/consolidated_summary.txt").read())
    print(f"{(time.time() - t0) / 60:.1f} min for one episode, including the model download and load")
else:
    print("resuming; timing check skipped")

## 8. Run

Runs in the foreground so that Save & Run All waits for it. Prints progress every 10 minutes; the full log is
`evaluate_all.runner.log` in the output folder.

In [ ]:
cmd = (f"cd {REPO} && {PY} -u baseline_performance/evaluate_all.py --resume_dir {OUT_ROOT} --tasks {TASKS} "
       f"--seeds {SEEDS} --n_episodes {N_EPISODES} {EVAL_ARGS} --jobs {JOBS} "
       f"--gpus {GPU_IDS} --launch_deadline_min {LAUNCH_DEADLINE_MIN} --hard_stop_min {HARD_STOP_MIN}")
print("$", cmd)
log_path = f"{OUT_ROOT}/evaluate_all.runner.log"
with open(log_path, "a") as log:
    proc = subprocess.Popen(cmd, shell=True, executable="/bin/bash", env=RUN_ENV, stdout=log, stderr=subprocess.STDOUT)
    while proc.poll() is None:
        time.sleep(600)
        n = len(glob.glob(f"{OUT_ROOT}/task_*_seed_*/eval_info.json"))
        print(time.strftime("%H:%M"), f"{n} evaluations finished", flush=True)
print("\n".join(open(log_path).read().splitlines()[-15:]))
RUN_EXIT = proc.returncode
print("exit code", RUN_EXIT, "(2 = incomplete; add this version's output as input and run again)")

## 9. Results and comparisons

1. Per task: this run, the local `results_300926` (`HuggingFaceVLA/smolvla_libero`, same init states and
   seeds) and LIBERO-CTRL (`lerobot/smolvla_libero`, one rollout per init state).
2. Against `results_300926`, paired by (task, seed, init state), with `compare_per_state.py`.
3. Each seed against LIBERO-CTRL, paired by (task, init state).

How to read it for `CHECKPOINT = "lerobot"`: close to LIBERO-CTRL's 59/100 on these init states means our
evaluation path is sound and the gap to the published score comes from the released checkpoint. Close to
the local 40.7% means our evaluation path loses performance, and the harness has to be fixed first.


In [ ]:
print(open(f"{OUT_ROOT}/consolidated_summary.txt").read())
sys.path.insert(0, f"{REPO}/baseline_performance")
from compare_per_state import compare, format_report

def load(path, seed=None):
    with open(path) as f:
        rows = list(csv.DictReader(f))
    return {(int(r["task_id"]), int(r.get("seed", 0)) if seed is None else seed, int(r["init_state"])): int(r["success"])
            for r in rows}

here = load(f"{OUT_ROOT}/per_episode.csv")
local = load(f"{REPO}/baseline_performance/results_300926/per_episode.csv")
ctrl = load(f"{REPO}/baseline_performance/reference_libero_ctrl/smolvla_libero_10_clean.csv", seed=0)
states = range(N_EPISODES)

print("task | this run | local results_300926 | LIBERO-CTRL, same init states | LIBERO-CTRL, all 50")
for t in range(10):
    h = [v for k, v in here.items() if k[0] == t]
    l = [v for k, v in local.items() if k[0] == t and k in here]
    c = [ctrl[(t, 0, i)] for i in states]
    c50 = [v for k, v in ctrl.items() if k[0] == t]
    print(f"{t:4d} | {sum(h):3d}/{len(h):<3d} | {sum(l):3d}/{len(l):<3d} | {sum(c):2d}/{len(c)} | {sum(c50):2d}/{len(c50)}")
n_ctrl = sum(ctrl[(t, 0, i)] for t in range(10) for i in states)
print(f"all  | {sum(here.values())}/{len(here)} ({100 * sum(here.values()) / max(1, len(here)):.1f}%) | "
      f"LIBERO-CTRL same init states {n_ctrl}/100 | LIBERO-CTRL all 50: {sum(ctrl.values())}/{len(ctrl)}")

common = {k: local[k] for k in here if k in local}
if common and len(common) == len(here):
    print("\n--- against results_300926, same episodes ---")
    print(format_report(compare(common, here), "local results_300926", f"Kaggle {CKPT['repo']}"))
base_csv = f"{REPO}/week6/smolvla_libero_lerobot_eval_kaggle/per_episode.csv"
if CHECKPOINT == "continued":
    base = load(base_csv)
    assert set(here) == set(base), "run incomplete: the hypothesis test needs all 300 episodes"
    print("\n--- HYPOTHESIS TEST: against lerobot/smolvla_libero on this notebook (171/300), same episodes ---")
    report = format_report(compare(base, here), "lerobot/smolvla_libero (Kaggle)", "continued training")
    print(report)
    open(f"{OUT_ROOT}/comparison_with_smolvla_libero_lerobot_eval.txt", "w").write(report)
if CHECKPOINT == "lerobot" and N_ACTION_STEPS is not None and not RUN_TAG:
    print(f"\n--- noise_reachability step 0: n_action_steps={N_ACTION_STEPS} against n_action_steps=1, same episodes ---")
    sh(f"{PY} -S {REPO}/noise_reachability/compare_action_steps.py {base_csv} {N_ACTION_STEPS}={OUT_ROOT}/per_episode.csv "
       f"--alpha 0.025 --out {OUT_ROOT}/comparison_with_n_action_steps_1.txt", check=False)
if RUN_TAG == "step1":
    print("\n--- noise_reachability step 1: pass@k over seeds 0-15 (seeds 0-2 from the repo) ---")
    sh(f"{PY} -S {REPO}/noise_reachability/reachability.py --tasks {TASKS} --seeds 0-15 "
       f"{REPO}/week6/noise_reachability_step0_kaggle/nas10_all_tasks/per_episode.csv {OUT_ROOT}/per_episode.csv "
       f"--out {OUT_ROOT}/reachability.txt", check=False)
for s in sorted({k[1] for k in here}):
    mine = {(t, 0, i): here[(t, s, i)] for t in range(10) for i in states if (t, s, i) in here}
    if len(mine) == 10 * len(states):
        print(f"\n--- seed {s} against LIBERO-CTRL, paired by (task, init state) ---")
        print(format_report(compare({k: ctrl[k] for k in mine}, mine), "LIBERO-CTRL", f"Kaggle seed {s}"))


## 10. Files for the repo

The output folder (with videos) is in the version's **Output** tab. This zip holds only the small files; unpack
it into `week6/baseline_lerobot_eval_kaggle/` in the repo.

In [ ]:
sh(f"cd '{OUT_ROOT}' && zip -q -r /kaggle/working/{OUT_NAME}_results.zip . -x '*/videos/*' '*.mp4'")
sh(f"du -h /kaggle/working/{OUT_NAME}_results.zip")